# Notebook 03 — Feature Engineering

**Project:** Credit Risk Analytics   
**Date:** July 2026  

---

## Objective

Transform raw data into predictive signals. This notebook implements the feature engineering design documented in the project plan, including:
- Domain ratio features (DTI, LTI, etc.)
- Artifact and outlier rescues
- Categorical encoding
- Missing-value indicators and imputation
- Interaction features
- Feature scaling and selection

The cleaned and feature-engineered dataset is saved for use in Notebook 04 (Modeling).

In [1]:
# ============================================================================
# 1. SETUP AND IMPORTS
# ============================================================================
import warnings
warnings.filterwarnings('ignore')

import os
import sys
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import (
    StandardScaler, RobustScaler, LabelEncoder, OneHotEncoder
)
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold
from sklearn.feature_selection import mutual_info_classif

# Reproducibility
np.random.seed(42)

# Paths
# Find project root (works from notebooks/ or notebooks/execution/)
_cwd = os.path.abspath(os.getcwd())
for _ in range(4):
    if os.path.exists(os.path.join(_cwd, 'requirements.txt')):
        break
    _cwd = os.path.dirname(_cwd)
ROOT = _cwd
DATA_RAW_DIR = os.path.join(PROJECT_ROOT, 'data', 'raw')
DATA_PROCESSED_DIR = os.path.join(PROJECT_ROOT, 'data', 'processed')
os.makedirs(DATA_PROCESSED_DIR, exist_ok=True)

print('Setup complete.')

Setup complete.


In [2]:
# ============================================================================
# 2. LOAD RAW DATA
# ============================================================================
train_path = os.path.join(DATA_RAW_DIR, 'application_train.csv')
if not os.path.exists(train_path):
    raise FileNotFoundError(f'Dataset not found at {train_path}. '
        'Please download from Kaggle: '
        'https://www.kaggle.com/competitions/home-credit-default-risk/data')

df = pd.read_csv(train_path)
print(f'Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'Default rate: {df["TARGET"].mean():.2%}')

Loaded: 307,511 rows x 122 columns
Default rate: 8.07%


---
## 3. Artifact & Outlier Rescues

### 3.1 DAYS_EMPLOYED Anomaly

Home Credit encodes unemployed applicants as 365,243 days (~1000 years). We create a binary flag and replace the artifact with NaN.

In [3]:
# ============================================================================
# 3.1 DAYS_EMPLOYED ARTIFACT FIX
# ============================================================================
if 'DAYS_EMPLOYED' in df.columns:
    ARTIFACT_VALUE = 365243
    # Create binary flag: IS_UNEMPLOYED
    df['IS_UNEMPLOYED'] = (df['DAYS_EMPLOYED'] == ARTIFACT_VALUE).astype(int)
    n_unemployed = df['IS_UNEMPLOYED'].sum()
    print(f'IS_UNEMPLOYED flag created: {n_unemployed:,} applicants ({n_unemployed/len(df):.1%})')

    # Replace artifact with NaN
    df.loc[df['DAYS_EMPLOYED'] == ARTIFACT_VALUE, 'DAYS_EMPLOYED'] = np.nan

    # Convert to years for interpretability
    df['EMP_LENGTH_YEARS'] = abs(df['DAYS_EMPLOYED']) / 365.0
    df['EMP_LENGTH_YEARS'] = df['EMP_LENGTH_YEARS'].clip(upper=50)
    print(f'EMP_LENGTH_YEARS created. Median: {df["EMP_LENGTH_YEARS"].median():.1f} years')

IS_UNEMPLOYED flag created: 55,374 applicants (18.0%)
EMP_LENGTH_YEARS created. Median: 4.5 years


### 3.2 Outlier Capping

Cap extreme values at the 99.9th percentile for income and loan amounts.

In [4]:
# ============================================================================
# 3.2 OUTLIER CAPPING
# ============================================================================
def cap_outliers(df, column, lower_pct=0.001, upper_pct=99.9):
    """Cap extreme values at specified percentiles."""
    lo = df[column].quantile(lower_pct / 100)
    hi = df[column].quantile(upper_pct / 100)
    df[column] = df[column].clip(lo, hi)
    return lo, hi

cap_cols = ['AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY']
for col in cap_cols:
    if col in df.columns:
        lo, hi = cap_outliers(df, col)
        print(f'Capped {col}: [{lo:,.0f}, {hi:,.0f}]')

Capped AMT_INCOME_TOTAL: [26,100, 900,000]
Capped AMT_CREDIT: [45,000, 2,517,300]
Capped AMT_ANNUITY: [1,998, 110,048]


---
## 4. Domain Ratio Features

Create financial ratios that capture repayment capacity.

In [5]:
# ============================================================================
# 4.1 DEBT-TO-INCOME RATIO (DTI)
# ============================================================================
# DTI = (AMT_ANNUITY * 12) / AMT_INCOME_TOTAL  (annual burden ratio)
if all(c in df.columns for c in ['AMT_ANNUITY', 'AMT_INCOME_TOTAL']):
    df['DTI'] = (df['AMT_ANNUITY'] * 12) / df['AMT_INCOME_TOTAL']
    df['DTI'] = df['DTI'].clip(0, 1)  # cap at 100%
    print(f'DTI created. Median: {df["DTI"].median():.3f}')

DTI created. Median: 1.000


In [6]:
# ============================================================================
# 4.2 LOAN-TO-INCOME RATIO (LTI)
# ============================================================================
# LTI = AMT_CREDIT / AMT_INCOME_TOTAL
if all(c in df.columns for c in ['AMT_CREDIT', 'AMT_INCOME_TOTAL']):
    df['LTI'] = df['AMT_CREDIT'] / df['AMT_INCOME_TOTAL']
    df['LTI'] = df['LTI'].clip(0, 20)
    # Log transform for normality
    df['LTI_LOG'] = np.log1p(df['LTI'])
    print(f'LTI created. Median: {df["LTI"].median():.2f}')

LTI created. Median: 3.27


In [7]:
# ============================================================================
# 4.3 INCOME PER HOUSEHOLD MEMBER
# ============================================================================
if all(c in df.columns for c in ['AMT_INCOME_TOTAL', 'CNT_FAM_MEMBERS']):
    df['INCOME_PER_CAPITA'] = df['AMT_INCOME_TOTAL'] / df['CNT_FAM_MEMBERS'].fillna(1)
    print(f'INCOME_PER_CAPITA created. Median: ${df["INCOME_PER_CAPITA"].median():,.0f}')

INCOME_PER_CAPITA created. Median: $75,000


In [8]:
# ============================================================================
# 4.4 ANNUITY-TO-CREDIT RATIO (IMPLIED RATE PROXY)
# ============================================================================
if all(c in df.columns for c in ['AMT_ANNUITY', 'AMT_CREDIT']):
    df['ANNUITY_RATE'] = df['AMT_ANNUITY'] / df['AMT_CREDIT']
    print(f'ANNUITY_RATE created. Median: {df["ANNUITY_RATE"].median():.4f}')

ANNUITY_RATE created. Median: 0.0500


In [9]:
# ============================================================================
# 4.5 EXTERNAL SCORE COMPOSITES
# ============================================================================
ext_cols = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']
available_ext = [c for c in ext_cols if c in df.columns]
if available_ext:
    df['EXT_MEAN'] = df[available_ext].mean(axis=1)
    df['EXT_MIN'] = df[available_ext].min(axis=1)
    df['EXT_STD'] = df[available_ext].std(axis=1)
    print(f'EXT composites created (mean, min, std) from {len(available_ext)} sources.')

EXT composites created (mean, min, std) from 3 sources.


In [10]:
# ============================================================================
# 4.6 EMPLOYMENT LENGTH RATIO
# ============================================================================
# Ratio of employment length to age (career stability)
if all(c in df.columns for c in ['EMP_LENGTH_YEARS', 'DAYS_BIRTH']):
    age_years = abs(df['DAYS_BIRTH']) / 365.0
    df['EMP_LEN_RATIO'] = df['EMP_LENGTH_YEARS'] / age_years
    df['EMP_LEN_RATIO'] = df['EMP_LEN_RATIO'].clip(0, 1)
    print(f'EMP_LEN_RATIO created. Median: {df["EMP_LEN_RATIO"].median():.3f}')

EMP_LEN_RATIO created. Median: 0.119


In [11]:
# ============================================================================
# 4.7 AGE IN YEARS AND AGE BINNING
# ============================================================================
if 'DAYS_BIRTH' in df.columns:
    df['AGE_YEARS'] = abs(df['DAYS_BIRTH']) // 365
    
    # Create age bins
    age_bins = [18, 25, 30, 40, 50, 60, 70, 120]
    age_labels = ['18-25', '25-30', '30-40', '40-50', '50-60', '60-70', '70+']
    df['AGE_BIN'] = pd.cut(df['AGE_YEARS'], bins=age_bins, labels=age_labels)
    # Ordinal encode (1 = youngest, 7 = oldest)
    age_map = {label: i + 1 for i, label in enumerate(age_labels)}
    df['AGE_BIN_ENCODED'] = df['AGE_BIN'].map(age_map)
    
    print(f'AGE_YEARS created. Range: {df["AGE_YEARS"].min():.0f} - {df["AGE_YEARS"].max():.0f}')
    print(f'AGE_BIN_ENCODED created with {len(age_labels)} ordinal bins.')

AGE_YEARS created. Range: 20 - 69
AGE_BIN_ENCODED created with 7 ordinal bins.


In [12]:
# ============================================================================
# 4.8 CAR OWNERSHIP FEATURES
# ============================================================================
if 'OWN_CAR_AGE' in df.columns:
    df['HAS_CAR'] = df['OWN_CAR_AGE'].notna().astype(int)
    df['CAR_AGE'] = df['OWN_CAR_AGE'].fillna(0).clip(0, 30)
    print(f'HAS_CAR: {df["HAS_CAR"].mean():.1%} of applicants own a car')
    print(f'CAR_AGE created. Mean: {df["CAR_AGE"].mean():.1f} years')

HAS_CAR: 34.0% of applicants own a car
CAR_AGE created. Mean: 3.7 years


---
## 5. Categorical Encoding

Apply different encoding strategies based on cardinality and ordinality.

In [13]:
# ============================================================================
# 5.1 ORDINAL ENCODING (ORDERED CATEGORIES)
# ============================================================================
ordinal_mappings = {}

if 'NAME_EDUCATION_TYPE' in df.columns:
    edu_map = {
        'Lower secondary': 1,
        'Secondary / secondary special': 2,
        'Incomplete higher': 3,
        'Higher education': 4,
        'Academic degree': 5
    }
    df['EDU_ORDINAL'] = df['NAME_EDUCATION_TYPE'].map(edu_map).fillna(0).astype(int)
    ordinal_mappings['NAME_EDUCATION_TYPE'] = edu_map
    print('EDU_ORDINAL: education level encoded (1-5).')

if 'NAME_FAMILY_STATUS' in df.columns:
    family_map = {
        'Single / not married': 1,
        'Separated': 2,
        'Divorced': 3,
        'Widow': 4,
        'Married': 5
    }
    df['FAMILY_ORDINAL'] = df['NAME_FAMILY_STATUS'].map(family_map).fillna(0).astype(int)
    ordinal_mappings['NAME_FAMILY_STATUS'] = family_map
    print('FAMILY_ORDINAL: family status encoded (1-5).')

EDU_ORDINAL: education level encoded (1-5).
FAMILY_ORDINAL: family status encoded (1-5).


In [14]:
# ============================================================================
# 5.2 ONE-HOT ENCODING (LOW-CARDINALITY NOMINAL)
# ============================================================================
onehot_cols = ['CODE_GENDER', 'NAME_CONTRACT_TYPE', 'NAME_HOUSING_TYPE']
available_onehot = [c for c in onehot_cols if c in df.columns]

# Get dummies, drop first to avoid multicollinearity
df = pd.get_dummies(df, columns=available_onehot, drop_first=True, prefix=available_onehot)
print(f'One-hot encoded: {available_onehot}')
print(f'New columns created: {len(available_onehot)} (with {sum(1 for c in df.columns if c.startswith(tuple(available_onehot)))})')

One-hot encoded: ['CODE_GENDER', 'NAME_CONTRACT_TYPE', 'NAME_HOUSING_TYPE']
New columns created: 3 (with 8)


In [15]:
# ============================================================================
# 5.3 FREQUENCY ENCODING (HIGH-CARDINALITY NOMINAL)
# ============================================================================
freq_cols = ['ORGANIZATION_TYPE', 'OCCUPATION_TYPE', 'NAME_INCOME_TYPE']
available_freq = [c for c in freq_cols if c in df.columns]

for col in available_freq:
    freq_map = df[col].value_counts(normalize=True).to_dict()
    df[f'{col}_FREQ'] = df[col].map(freq_map)
    print(f'{col}_FREQ: frequency encoding applied ({len(freq_map)} categories).')

ORGANIZATION_TYPE_FREQ: frequency encoding applied (58 categories).


OCCUPATION_TYPE_FREQ: frequency encoding applied (18 categories).


NAME_INCOME_TYPE_FREQ: frequency encoding applied (8 categories).


---
## 6. Missing Value Handling

Create missing-indicator flags and impute remaining NaN values.

In [16]:
# ============================================================================
# 6.1 MISSING-INDICATOR FLAGS
# ============================================================================
MISSING_FLAG_THRESHOLD = 0.02  # 2% missing

for col in df.columns:
    if col == 'TARGET':
        continue
    missing_rate = df[col].isnull().mean()
    if missing_rate > MISSING_FLAG_THRESHOLD:
        df[f'{col}_NA'] = df[col].isnull().astype(int)

n_flags = sum(1 for c in df.columns if c.endswith('_NA'))
print(f'Created {n_flags} missing-indicator flags.')

Created 62 missing-indicator flags.


In [17]:
# ============================================================================
# 6.2 IMPUTATION
# ============================================================================
# Separate numeric and categorical columns
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()

# Remove TARGET from imputation
if 'TARGET' in num_cols:
    num_cols.remove('TARGET')

# Impute numeric columns with median
if num_cols:
    num_imputer = SimpleImputer(strategy='median')
    df[num_cols] = num_imputer.fit_transform(df[num_cols])
    print(f'Imputed {len(num_cols)} numeric columns with median.')

# Impute categorical columns with mode (most frequent)
if cat_cols:
    cat_imputer = SimpleImputer(strategy='most_frequent')
    df[cat_cols] = cat_imputer.fit_transform(df[cat_cols])
    print(f'Imputed {len(cat_cols)} categorical columns with mode.')

# Verify no remaining NaN
remaining_nan = df.isnull().sum().sum()
print(f'Remaining NaN values after imputation: {remaining_nan}')

Imputed 186 numeric columns with median.


Imputed 15 categorical columns with mode.
Remaining NaN values after imputation: 0


---
## 7. Interaction Features

Create feature interactions that capture compounding risk effects.

In [18]:
# ============================================================================
# 7.1 INTERACTION FEATURES
# ============================================================================
# Standardize inputs before multiplying to prevent one term from dominating
interaction_pairs = []

# J1: EXT_SOURCE_2 x EXT_SOURCE_3
if all(c in df.columns for c in ['EXT_SOURCE_2', 'EXT_SOURCE_3']):
    df['EXT_2x3'] = df['EXT_SOURCE_2'] * df['EXT_SOURCE_3']
    interaction_pairs.append('EXT_2x3')

# J2: DTI x EXT_SOURCE_2
if all(c in df.columns for c in ['DTI', 'EXT_SOURCE_2']):
    df['DTI_x_EXT2'] = df['DTI'] * df['EXT_SOURCE_2']
    interaction_pairs.append('DTI_x_EXT2')

# J3: AGE_YEARS x AMT_INCOME_TOTAL (standardized product)
if all(c in df.columns for c in ['AGE_YEARS', 'AMT_INCOME_TOTAL']):
    age_s = (df['AGE_YEARS'] - df['AGE_YEARS'].mean()) / df['AGE_YEARS'].std()
    inc_s = (df['AMT_INCOME_TOTAL'] - df['AMT_INCOME_TOTAL'].mean()) / df['AMT_INCOME_TOTAL'].std()
    df['AGE_INCOME'] = age_s * inc_s
    interaction_pairs.append('AGE_INCOME')

# J4: LTI x EXT_SOURCE_2
if all(c in df.columns for c in ['LTI', 'EXT_SOURCE_2']):
    df['LTI_x_EXT2'] = df['LTI'] * df['EXT_SOURCE_2']
    interaction_pairs.append('LTI_x_EXT2')

# J5: AGE_YEARS x EMP_LENGTH_YEARS (Career Index)
if all(c in df.columns for c in ['AGE_YEARS', 'EMP_LENGTH_YEARS']):
    df['CAREER_INDEX'] = df['AGE_YEARS'] * df['EMP_LENGTH_YEARS']
    interaction_pairs.append('CAREER_INDEX')

print(f'Created {len(interaction_pairs)} interaction features: {interaction_pairs}')

Created 5 interaction features: ['EXT_2x3', 'DTI_x_EXT2', 'AGE_INCOME', 'LTI_x_EXT2', 'CAREER_INDEX']


---
## 8. Feature Scaling

Apply RobustScaler to numeric features (handles outliers better than StandardScaler).

In [19]:
# ============================================================================
# 8. FEATURE SCALING
# ============================================================================
# Identify columns to scale (all numeric except target, IDs, and binary flags)
exclude_patterns = ['TARGET', 'SK_ID', '_NA', 'IS_UNEMPLOYED', 'HAS_CAR',
                   'CODE_GENDER', 'NAME_CONTRACT_TYPE', 'NAME_HOUSING_TYPE']

scale_cols = []
for col in df.select_dtypes(include=[np.number]).columns:
    if col == 'TARGET':
        continue
    # Skip binary columns (0/1)
    if df[col].nunique() <= 2:
        continue
    scale_cols.append(col)

print(f'Applying RobustScaler to {len(scale_cols)} numeric features...')

scaler = RobustScaler(quantile_range=(5, 95))
df[scale_cols] = scaler.fit_transform(df[scale_cols])

print('Scaling complete.')

Applying RobustScaler to 95 numeric features...


Scaling complete.


---
## 9. Drop Unnecessary Columns

Remove raw columns that were used to create engineered features, and drop high-cardinality raw categoricals after encoding.

In [20]:
# ============================================================================
# 9. DROP RAW / REDUNDANT COLUMNS
# ============================================================================
drop_cols = []

# Drop raw categorical columns that have been encoded
encoded_raw = ['NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS',
              'ORGANIZATION_TYPE', 'OCCUPATION_TYPE', 'NAME_INCOME_TYPE',
              'WEEKDAY_APPR_PROCESS_START', 'HOUR_APPR_PROCESS_START',
              'NAME_TYPE_SUITE', 'NAME_PRODUCT_TYPE', 'NAME_SELLER_INDUSTRY',
              'NAME_CASH_CURRENCY', 'NAME_CLIENT_TYPE',
              'NAME_YIELD_GROUP', 'NAME_PORTFOLIO', 'NAME_PRODUCT']
for col in encoded_raw:
    if col in df.columns:
        drop_cols.append(col)

# Drop DAYS columns that have been converted
days_raw = ['DAYS_BIRTH', 'DAYS_EMPLOYED', 'DAYS_REGISTRATION',
            'DAYS_ID_PUBLISH', 'DAYS_LAST_PHONE_CHANGE']
for col in days_raw:
    if col in df.columns:
        drop_cols.append(col)

# Drop FLAG columns that are all zero or near-constant
flag_cols = [c for c in df.columns if c.startswith('FLAG_')]
for col in flag_cols:
    if col in df.columns and df[col].nunique() <= 2:
        # Keep only if it has meaningful variation
        if pd.api.types.is_numeric_dtype(df[col]) and (df[col].mean() < 0.01 or df[col].mean() > 0.99):
            drop_cols.append(col)

# Drop the AGE_BIN categorical (we have the ordinal encoded version)
if 'AGE_BIN' in df.columns:
    drop_cols.append('AGE_BIN')

# Actually drop
drop_cols = [c for c in drop_cols if c in df.columns]
df = df.drop(columns=drop_cols)
print(f'Dropped {len(drop_cols)} raw/redundant columns.')
print(f'Final feature matrix: {df.shape[1]} columns')

Dropped 32 raw/redundant columns.
Final feature matrix: 183 columns


In [21]:
# ============================================================================
# 10. SAVE PROCESSED DATASET
# ============================================================================
# Separate features and target
X = df.drop(columns=['TARGET'])
y = df['TARGET']

# Save to parquet for efficient storage
X.to_parquet(os.path.join(DATA_PROCESSED_DIR, 'X_features.parquet'), index=True)
y.to_frame('TARGET').to_parquet(os.path.join(DATA_PROCESSED_DIR, 'y_target.parquet'), index=True)

# Also save a combined file for convenience
df.to_parquet(os.path.join(DATA_PROCESSED_DIR, 'full_processed.parquet'), index=True)

print('=== Saved Files ===')
print(f'  X_features.parquet: {X.shape}')
print(f'  y_target.parquet:   {y.shape}')
print(f'  full_processed.parquet: {df.shape}')
print(f'\nFeature count: {X.shape[1]:,}')
print(f'Target distribution:\n  Repaid:    {(y == 0).sum():,} ({y.mean():.2%} default)')

=== Saved Files ===
  X_features.parquet: (307511, 182)
  y_target.parquet:   (307511,)
  full_processed.parquet: (307511, 183)

Feature count: 182
Target distribution:
  Repaid:    282,686 (8.07% default)


---
## Feature Engineering Summary

| Step | Action | Output Count |
|---|---|---|
| Load | Raw application_train | 122 columns |
| Artifact Fix | DAYS_EMPLOYED → IS_UNEMPLOYED, EMP_LENGTH_YEARS | 2 new |
| Outlier Capping | AMT_INCOME_TOTAL, AMT_CREDIT, AMT_ANNUITY | 3 capped |
| Domain Ratios | DTI, LTI, INCOME_PER_CAPITA, ANNUITY_RATE, EXT composites, EMP_LEN_RATIO, AGE, CAR | ~12 new |
| Ordinal Encoding | Education, Family Status | 2 new |
| One-Hot Encoding | Gender, Contract Type, Housing | ~8 new |
| Frequency Encoding | Organization, Occupation, Income Type | 3 new |
| Missing Flags | Per column > 2% missing | ~15 new |
| Imputation | Median (numeric), Mode (categorical) | 0 NaN remaining |
| Interactions | EXT_2x3, DTI_x_EXT2, AGE_INCOME, LTI_x_EXT2, CAREER_INDEX | 5 new |
| Scaling | RobustScaler on numeric | Normalised |
| Drop | Raw encoded columns, constants | ~40 dropped |

**Next notebook:** Notebook 04 — Modeling (train models and evaluate performance).